# Individual Activity Exploration

This notebook contains individual exploration of the Google Local New York dataset to investigate how activity-related places and user interactions can be defined and filtered.

**Branch:** `ogechika/activity-exploration`

In [1]:
import pandas as pd
from pathlib import Path
from collections import Counter

In [2]:
PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
elif (PROJECT_DIR / "Activity-Recommendation-System").exists():
    PROJECT_DIR = PROJECT_DIR / "Activity-Recommendation-System"

DATA_DIR = PROJECT_DIR / "data" / "raw"

print("Project directory:", PROJECT_DIR)
print("Data directory:", DATA_DIR)
print("Data directory exists:", DATA_DIR.exists())

Project directory: /Users/ogechikaesedebe/Downloads/Activity-Recommendation-System
Data directory: /Users/ogechikaesedebe/Downloads/Activity-Recommendation-System/data/raw
Data directory exists: True


In [3]:
metadata_files = list(DATA_DIR.glob("meta*.json.gz"))
review_files = list(DATA_DIR.glob("review*.json"))

print("Metadata files:", metadata_files)
print("Review files:", review_files)

Metadata files: [PosixPath('/Users/ogechikaesedebe/Downloads/Activity-Recommendation-System/data/raw/meta-New_York.json.gz')]
Review files: [PosixPath('/Users/ogechikaesedebe/Downloads/Activity-Recommendation-System/data/raw/review-New_York_10 2.json')]


In [4]:
if len(metadata_files) != 1:
    raise FileNotFoundError(
        "Expected one metadata file in data/raw/. "
        "Please check the dataset setup instructions."
    )

if len(review_files) != 1:
    raise FileNotFoundError(
        "Expected one review file in data/raw/. "
        "Please check the dataset setup instructions."
    )

metadata_path = metadata_files[0]
review_path = review_files[0]

print("Metadata:", metadata_path.name)
print("Reviews:", review_path.name)

Metadata: meta-New_York.json.gz
Reviews: review-New_York_10 2.json


In [5]:
metadata = pd.read_json(
    metadata_path,
    lines=True,
    compression="gzip"
)

print("Metadata shape:", metadata.shape)
metadata.head()

Metadata shape: (272189, 15)


,name,address,gmap_id,description,latitude,longitude,category,avg_rating,num_of_reviews,price,hours,MISC,state,relative_results,url
0,A-Top Insurance,"A-Top Insurance, 1009 Brighton Beach Ave, Broo...",0x89c24469c758686b:0x641f5b84cb9bedfa,None,40.578254,-73.959127,"[Insurance broker, Insurance agency]",2.0,4,None,"[[Thursday, 10AM–6PM], [Friday, 10AM–6PM], [Sa...",None,Open ⋅ Closes 6PM,"[0x89c24449907718fb:0x31b554a0983f621d, 0x4065...",https://www.google.com/maps/place//data=!4m2!3...
1,T-Mobile,"T-Mobile, 3923 103rd St, Queens, NY 11368",0x89c25fc9494dce47:0x6d63c807b59a55,None,40.750146,-73.862536,"[Cell phone store, Electronic parts supplier, ...",3.5,95,$$,"[[Thursday, 10AM–8PM], [Friday, 10AM–8PM], [Sa...","{'Service options': ['In-store shopping', 'Del...",Open ⋅ Closes 8PM,"[0x89c25fc7a91c609f:0xb103d6a261373fd, 0x89c25...",https://www.google.com/maps/place//data=!4m2!3...
2,Ace Hardware,"Ace Hardware, 130 4th Ave, New York, NY 10003",0x89c259992463b6e1:0x4a601a43752541dd,"Chain retailer stocking household tools, suppl...",40.733182,-73.990201,"[Hardware store, Building materials store, Ele...",3.4,8,None,"[[Thursday, 8AM–6:30PM], [Friday, 8AM–6:30PM],...",None,Permanently closed,"[0x89c2598340fb6509:0xa996f3db49ba6e78, 0x89c2...",https://www.google.com/maps/place//data=!4m2!3...
3,"Agricultural Consulting Services, Inc.","Agricultural Consulting Services, Inc., 370 Su...",0x89d14bf74320aea5:0x720261cd8960f8cd,None,43.054743,-77.649136,[Corporate office],3.0,1,None,"[[Thursday, 9AM–5PM], [Friday, 9AM–5PM], [Satu...",None,Open ⋅ Closes 5PM,None,https://www.google.com/maps/place//data=!4m2!3...
4,Long Point,"Long Point, Ledyard, NY 13026",0x89d0bd8d2df8d10f:0x77a8f349f40492fa,None,42.715626,-76.710227,[Peninsula],5.0,1,None,None,None,None,None,"https://www.google.com/maps/place/Ledyard,+NY+..."


In [6]:
# Load an exploratory sample of reviews
reviews_sample = pd.read_json(
    review_path,
    lines=True,
    nrows=100000
)

print("Review sample shape:", reviews_sample.shape)
reviews_sample.head()

Review sample shape: (100000, 8)


,user_id,name,time,rating,text,pics,resp,gmap_id
0,1.137221e+20,Alvin Martinez,1603494795361,5,I'm late to posting this but this store especi...,[{'url': ['https://lh5.googleusercontent.com/p...,None,0x89c25fc9494dce47:0x6d63c807b59a55
1,1.072934e+20,Johnnie Jackson,1620157037403,1,Very dissatisfied I did not get my phone the p...,None,"{'time': 1620268360920, 'text': 'We pride ours...",0x89c25fc9494dce47:0x6d63c807b59a55
2,1.003786e+20,Manie Blazer,1597431662039,5,Excellent very well done with professional car...,None,None,0x89c25fc9494dce47:0x6d63c807b59a55
3,1.149982e+20,Fashion Fiinds,1543773862044,5,Basing my review strictly on the service I rec...,None,"{'time': 1543855317372, 'text': 'Thanks for th...",0x89c25fc9494dce47:0x6d63c807b59a55
4,1.171782e+20,Andres Rieloff,1597279097718,1,Bad! Disorganized. I'm being totally honest. I...,None,None,0x89c25fc9494dce47:0x6d63c807b59a55


In [7]:
category_counts = Counter()

for categories in metadata["category"].dropna():
    if isinstance(categories, list):
        category_counts.update(categories)

category_df = pd.DataFrame(
    category_counts.items(),
    columns=["category", "number_of_places"]
).sort_values(
    "number_of_places",
    ascending=False
).reset_index(drop=True)

print("Number of unique categories:", len(category_df))

category_df.head(50)

Number of unique categories: 3741


,category,number_of_places
0,Restaurant,19316
1,Bar,7925
2,Fast food restaurant,6648
3,Auto repair shop,6334
4,Pizza restaurant,6223
5,Coffee shop,5996
6,Delivery Restaurant,5811
7,Grocery store,5726
8,Beauty salon,5435
9,Takeout Restaurant,5433


In [8]:
activity_terms = [
    "park",
    "museum",
    "theater",
    "theatre",
    "cinema",
    "movie",
    "tour",
    "attraction",
    "zoo",
    "aquarium",
    "gallery",
    "art",
    "music",
    "concert",
    "dance",
    "bowling",
    "arcade",
    "game",
    "escape",
    "gym",
    "fitness",
    "yoga",
    "pilates",
    "climbing",
    "sports",
    "swimming",
    "pool",
    "skating",
    "ski",
    "golf",
    "tennis",
    "hiking",
    "trail",
    "beach",
    "garden",
    "historic",
    "amusement",
    "recreation"
]

pattern = "|".join(activity_terms)

candidate_categories = category_df[
    category_df["category"].str.contains(
        pattern,
        case=False,
        na=False
    )
].copy()

print("Candidate category labels:", len(candidate_categories))

candidate_categories.head(100)

Candidate category labels: 390


,category,number_of_places
14,Tourist attraction,4825
15,Park,4682
45,Auto parts store,2093
55,Gym,1860
100,Art gallery,1089
...,...,...
908,Sightseeing tour agency,105
918,Fishing charter,104
922,City park,103
927,Performing arts group,102


In [9]:
pd.set_option("display.max_rows", None)

candidate_categories[
    ["category", "number_of_places"]
].reset_index(drop=True)

,category,number_of_places
0,Tourist attraction,4825
1,Park,4682
2,Auto parts store,2093
3,Gym,1860
4,Art gallery,1089
5,Department store,1086
6,Fitness center,1053
7,Sports bar,864
8,Physical fitness program,860
9,Garden center,840


In [10]:
activity_taxonomy = {

    "outdoor_nature": [
        "Park",
        "State park",
        "City park",
        "National park",
        "Wildlife park",
        "Hiking area",
        "Beach",
        "Public beach",
        "Botanical garden",
        "Community garden",
        "Garden",
        "Swimming lake",
        "Dog park"
    ],

    "arts_culture": [
        "Museum",
        "Art museum",
        "History museum",
        "Science museum",
        "Local history museum",
        "Heritage museum",
        "Children's museum",
        "Historical place museum",
        "Art gallery",
        "Art center",
        "Historical landmark",
        "Historical place"
    ],

    "entertainment": [
        "Movie theater",
        "Drive-in movie theater",
        "Performing arts theater",
        "Drama theater",
        "Dinner theater",
        "Concert hall",
        "Live music venue",
        "Amphitheater",
        "Bowling alley",
        "Video arcade",
        "Escape room center",
        "Amusement center",
        "Children's amusement center",
        "Amusement park",
        "Theme park",
        "Water park",
        "Go-kart track",
        "Pool hall"
    ],

    "fitness_sports": [
        "Gym",
        "Fitness center",
        "Yoga studio",
        "Pilates studio",
        "Boxing gym",
        "Muay Thai boxing gym",
        "Rock climbing gym",
        "Gymnastics center",
        "Gymnastics club",
        "Sports complex",
        "Sports club",
        "Sports activity location",
        "Tennis court",
        "Tennis club",
        "Golf course",
        "Public golf course",
        "Private golf course",
        "Indoor golf course",
        "Miniature golf course",
        "Golf driving range",
        "Disc golf course",
        "Swimming pool",
        "Public swimming pool",
        "Indoor swimming pool",
        "Outdoor swimming pool",
        "Swimming facility",
        "Ice skating rink",
        "Roller skating rink",
        "Skateboard park"
    ],

    "attractions_tours": [
        "Tourist attraction",
        "Zoo",
        "Aquarium",
        "Tour operator",
        "Boat tour agency",
        "Bus tour agency",
        "Sightseeing tour agency",
        "Fishing charter"
    ],

    "classes_experiences": [
        "Martial arts school",
        "Dance school",
        "Music school",
        "Art school",
        "Swimming school",
        "Sports school",
        "Ballroom dance instructor",
        "Golf instructor",
        "Tennis instructor",
        "Swimming instructor",
        "Yoga instructor",
        "Adventure sports",
        "Adventure sports center"
    ]
}

In [11]:
all_categories = set(category_df["category"])

for group, categories in activity_taxonomy.items():
    found = [c for c in categories if c in all_categories]
    missing = [c for c in categories if c not in all_categories]

    print(f"\n{group.upper()}")
    print("Found:", len(found))
    print("Not found:", len(missing))

    if missing:
        print("Missing categories:", missing)


OUTDOOR_NATURE
Found: 13
Not found: 0

ARTS_CULTURE
Found: 12
Not found: 0

ENTERTAINMENT
Found: 18
Not found: 0

FITNESS_SPORTS
Found: 29
Not found: 0

ATTRACTIONS_TOURS
Found: 8
Not found: 0

CLASSES_EXPERIENCES
Found: 13
Not found: 0


In [12]:
# Combine all taxonomy categories into one set
activity_categories = {
    category
    for categories in activity_taxonomy.values()
    for category in categories
}

print("Number of activity category labels:", len(activity_categories))

Number of activity category labels: 93


In [13]:
def is_activity(categories):
    if not isinstance(categories, list):
        return False
    
    return any(
        category in activity_categories
        for category in categories
    )


activity_metadata = metadata[
    metadata["category"].apply(is_activity)
].copy()

print("Total places:", len(metadata))
print("Candidate activity places:", len(activity_metadata))

print(
    "Percentage classified as activities:",
    round(len(activity_metadata) / len(metadata) * 100, 2),
    "%"
)

Total places: 272189
Candidate activity places: 18104
Percentage classified as activities: 6.65 %


In [14]:
activity_metadata[
    [
        "name",
        "category",
        "avg_rating",
        "num_of_reviews"
    ]
].sample(20, random_state=42)

,name,category,avg_rating,num_of_reviews
100474,Woodstock Yoga Center,[Yoga studio],5.0,5
250144,Tanner Park,[Park],4.6,58
259675,Allerton Playground,"[Playground, Park]",4.2,87
85193,VIP Fit Club,"[Gym, Body shaping class, Boot camp, Boxing gy...",4.9,58
127304,Edward Boynton House - Frank Lloyd Wright,[Historical landmark],4.8,28
137175,Buffalo Academy for Visual and Performing Arts...,[Art school],4.2,28
44256,William Simmons Community Garden Club,[Garden],4.1,8
271396,National Museum of the American Indian,"[National museum, Museum, Tourist attraction]",4.4,3680
209962,Crunch Fitness - Greenpoint,"[Gym, Fitness center, Personal trainer, Physic...",3.4,194
271252,Marsha P. Johnson State Park,"[State park, City park, Cultural landmark, Par...",4.6,2638


In [15]:
# Get the gmap_ids for candidate activity places
activity_ids = set(activity_metadata["gmap_id"])

# Keep reviews associated with candidate activities
activity_reviews_sample = reviews_sample[
    reviews_sample["gmap_id"].isin(activity_ids)
].copy()

print("Reviews in exploratory sample:", len(reviews_sample))
print("Reviews for candidate activities:", len(activity_reviews_sample))

print(
    "Percentage of sample reviews linked to activities:",
    round(len(activity_reviews_sample) / len(reviews_sample) * 100, 2),
    "%"
)

Reviews in exploratory sample: 100000
Reviews for candidate activities: 5377
Percentage of sample reviews linked to activities: 5.38 %


In [16]:
print("Unique users:", activity_reviews_sample["user_id"].nunique())
print("Unique activity places:", activity_reviews_sample["gmap_id"].nunique())
print("Total activity reviews:", len(activity_reviews_sample))

Unique users: 4875
Unique activity places: 283
Total activity reviews: 5377


In [17]:
reviews_per_user = (
    activity_reviews_sample
    .groupby("user_id")
    .size()
    .sort_values(ascending=False)
)

print("Activity reviews per user:")
print(reviews_per_user.describe())

print("\nUsers with at least 2 activity reviews:",
      (reviews_per_user >= 2).sum())

print("Users with at least 3 activity reviews:",
      (reviews_per_user >= 3).sum())

print("Users with at least 5 activity reviews:",
      (reviews_per_user >= 5).sum())

print("Maximum activity reviews by one user:",
      reviews_per_user.max())

Activity reviews per user:
count    4875.000000
mean        1.102974
std         0.327982
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         6.000000
dtype: float64

Users with at least 2 activity reviews: 475
Users with at least 3 activity reviews: 20
Users with at least 5 activity reviews: 1
Maximum activity reviews by one user: 6


In [18]:
reviews_per_user.value_counts().sort_index().head(10)

1    4400
2     455
3      15
4       4
6       1
Name: count, dtype: int64

## Full Review Interaction Analysis

### Goal

The initial 100,000-review sample showed very sparse user–activity
interactions. However, this sample represents only a small portion of
the full New York review dataset.

This section examines a larger portion of the review data to determine:

1. How many reviews correspond to candidate activity places.
2. How many unique users interact with activity places.
3. How many unique activity places have interactions.
4. How many users have repeated activity interactions.
5. Whether the interaction data is sufficiently dense for recommendation.
6. What filtering criteria may be appropriate when constructing Dataset v1.

In [19]:
print("Review file:", review_path)
print("Candidate activity places:", len(activity_metadata))
print("Candidate activity IDs:", len(activity_ids))

Review file: /Users/ogechikaesedebe/Downloads/Activity-Recommendation-System/data/raw/review-New_York_10 2.json
Candidate activity places: 18104
Candidate activity IDs: 18034


In [20]:
print("Activity metadata rows:", len(activity_metadata))
print("Unique gmap_ids:", activity_metadata["gmap_id"].nunique())
print(
    "Duplicate gmap_id rows:",
    activity_metadata["gmap_id"].duplicated().sum()
)
print(
    "Missing gmap_ids:",
    activity_metadata["gmap_id"].isna().sum()
)

Activity metadata rows: 18104
Unique gmap_ids: 18034
Duplicate gmap_id rows: 70
Missing gmap_ids: 0


In [21]:
duplicate_activity_ids = activity_metadata[
    activity_metadata["gmap_id"].duplicated(keep=False)
].sort_values("gmap_id")

duplicate_activity_ids[
    ["gmap_id", "name", "category"]
].head(20)

,gmap_id,name,category
1731,0x4ccab7755367e6b3:0xfb5438273acb64bc,North Hudson Beach Park,[Park]
262,0x4ccab7755367e6b3:0xfb5438273acb64bc,North Hudson Beach Park,[Park]
1260,0x4ccaf93f0974c325:0x236fd0ed8e84c49f,Wilmington Recreation Park,[Park]
2729,0x4ccaf93f0974c325:0x236fd0ed8e84c49f,Wilmington Recreation Park,[Park]
2736,0x4ccb056a335054c9:0x2a41677a9e9a9002,Mt. Pisgah Recreation Center,[Park]
1267,0x4ccb056a335054c9:0x2a41677a9e9a9002,Mt. Pisgah Recreation Center,[Park]
1259,0x4ccb6a8c190838a3:0x28a8c623a6875ad,The Adirondack Sky Center & Observatory,"[Observatory, Tourist attraction]"
2728,0x4ccb6a8c190838a3:0x28a8c623a6875ad,The Adirondack Sky Center & Observatory,"[Observatory, Tourist attraction]"
2748,0x89c2430970427697:0x8ef9b1b58062cde8,Dolo's Fitness,[Gym]
1279,0x89c2430970427697:0x8ef9b1b58062cde8,Dolo's Fitness,[Gym]


### Duplicate Candidate Activity Records

The candidate activity metadata contains 18,104 rows representing 18,034
unique `gmap_id` values. There are 70 duplicate `gmap_id` rows and no missing
IDs.

Because reviews are linked to places using `gmap_id`, the interaction analysis
uses the 18,034 unique activity IDs. Duplicate metadata records therefore do
not cause reviews to be counted multiple times during review filtering.

The duplicate metadata records should be addressed later when constructing
the final cleaned activity dataset.

In [22]:
chunk_size = 250_000

total_reviews_processed = 0
total_activity_reviews = 0
chunk_results = []

for chunk_number, chunk in enumerate(
    pd.read_json(
        review_path,
        lines=True,
        chunksize=chunk_size
    ),
    start=1
):
    activity_chunk = chunk[
        chunk["gmap_id"].isin(activity_ids)
    ].copy()

    chunk_results.append(activity_chunk)

    total_reviews_processed += len(chunk)
    total_activity_reviews += len(activity_chunk)

    print(
        f"Chunk {chunk_number}: "
        f"{total_reviews_processed:,} reviews processed | "
        f"{total_activity_reviews:,} activity reviews found"
    )

Chunk 1: 250,000 reviews processed | 13,384 activity reviews found
Chunk 2: 500,000 reviews processed | 30,119 activity reviews found
Chunk 3: 750,000 reviews processed | 50,374 activity reviews found
Chunk 4: 1,000,000 reviews processed | 75,706 activity reviews found
Chunk 5: 1,250,000 reviews processed | 98,962 activity reviews found
Chunk 6: 1,500,000 reviews processed | 148,117 activity reviews found
Chunk 7: 1,750,000 reviews processed | 202,145 activity reviews found
Chunk 8: 2,000,000 reviews processed | 223,404 activity reviews found
Chunk 9: 2,250,000 reviews processed | 243,198 activity reviews found
Chunk 10: 2,500,000 reviews processed | 262,837 activity reviews found
Chunk 11: 2,750,000 reviews processed | 278,943 activity reviews found
Chunk 12: 3,000,000 reviews processed | 302,408 activity reviews found
Chunk 13: 3,250,000 reviews processed | 323,681 activity reviews found
Chunk 14: 3,500,000 reviews processed | 348,914 activity reviews found
Chunk 15: 3,750,000 review

In [23]:
activity_reviews_full = pd.concat(
    chunk_results,
    ignore_index=True
)

print("Activity review dataset shape:", activity_reviews_full.shape)
print("Total activity reviews:", f"{len(activity_reviews_full):,}")

Activity review dataset shape: (2221591, 8)
Total activity reviews: 2,221,591


In [24]:
print(
    "Unique users:",
    f"{activity_reviews_full['user_id'].nunique():,}"
)

print(
    "Unique reviewed activity places:",
    f"{activity_reviews_full['gmap_id'].nunique():,}"
)

print(
    "Candidate activity places:",
    f"{len(activity_ids):,}"
)

print(
    "Activity review coverage:",
    f"{activity_reviews_full['gmap_id'].nunique() / len(activity_ids) * 100:.2f}%"
)

Unique users: 504,136
Unique reviewed activity places: 12,041
Candidate activity places: 18,034
Activity review coverage: 66.77%


In [25]:
activities_per_user = (
    activity_reviews_full
    .groupby("user_id")["gmap_id"]
    .nunique()
)

print("Unique activity places reviewed per user:")
print(activities_per_user.describe())

print()
print("Users with at least 2 activities:", f"{(activities_per_user >= 2).sum():,}")
print("Users with at least 3 activities:", f"{(activities_per_user >= 3).sum():,}")
print("Users with at least 5 activities:", f"{(activities_per_user >= 5).sum():,}")
print("Users with at least 10 activities:", f"{(activities_per_user >= 10).sum():,}")
print("Users with at least 20 activities:", f"{(activities_per_user >= 20).sum():,}")
print("Maximum activities reviewed by one user:", activities_per_user.max())

Unique activity places reviewed per user:
count    504136.000000
mean          4.347743
std           5.342765
min           1.000000
25%           1.000000
50%           3.000000
75%           5.000000
max         343.000000
Name: gmap_id, dtype: float64

Users with at least 2 activities: 375,474
Users with at least 3 activities: 274,773
Users with at least 5 activities: 151,103
Users with at least 10 activities: 46,357
Users with at least 20 activities: 9,944
Maximum activities reviewed by one user: 343


##### Notes
-Our initial 100,000-review sample suggested extremely sparse activity histories, with most users appearing only once. Because the full review dataset contains approximately 18.66 million reviews, we expanded the analysis rather than drawing conclusions from the initial sample. Using the current provisional activity definition, the full review data produced approximately 2.22 million activity-related reviews across 504,136 users and 12,041 reviewed activity places. The median user reviewed 3 unique activity places, while approximately 54.5% of users reviewed at least 3. This indicates substantially more repeated interaction history than the initial sample suggested and supports further investigation of interaction-based recommendation methods.

## Activity-Side Interaction Analysis

Next, we examine how review interactions are distributed across candidate activity places. This helps determine whether interactions are concentrated among a small number of highly popular places or distributed across many activities.

In [26]:
users_per_activity = (
    activity_reviews_full
    .groupby("gmap_id")["user_id"]
    .nunique()
)

print("Unique users per reviewed activity:")
print(users_per_activity.describe())

print()
print("Activities with at least 2 users:", f"{(users_per_activity >= 2).sum():,}")
print("Activities with at least 5 users:", f"{(users_per_activity >= 5).sum():,}")
print("Activities with at least 10 users:", f"{(users_per_activity >= 10).sum():,}")
print("Activities with at least 20 users:", f"{(users_per_activity >= 20).sum():,}")
print("Activities with at least 50 users:", f"{(users_per_activity >= 50).sum():,}")
print("Activities with at least 100 users:", f"{(users_per_activity >= 100).sum():,}")
print("Maximum users for one activity:", users_per_activity.max())

Unique users per reviewed activity:
count    12041.000000
mean       182.032555
std        479.890930
min          5.000000
25%         22.000000
50%         50.000000
75%        138.000000
max       7516.000000
Name: user_id, dtype: float64

Activities with at least 2 users: 12,041
Activities with at least 5 users: 12,041
Activities with at least 10 users: 12,022
Activities with at least 20 users: 9,421
Activities with at least 50 users: 6,049
Activities with at least 100 users: 3,847
Maximum users for one activity: 7516


In [27]:
activity_metadata_unique = (
    activity_metadata
    .drop_duplicates(subset="gmap_id")
    .copy()
)

activity_popularity = (
    users_per_activity
    .rename("unique_users")
    .reset_index()
    .merge(
        activity_metadata_unique[
            ["gmap_id", "name", "category"]
        ],
        on="gmap_id",
        how="left"
    )
    .sort_values("unique_users", ascending=False)
)

activity_popularity.head(20)

,gmap_id,unique_users,name,category
2001,0x89c2599f2db306bb:0x2a5c7f9eea6938a0,7516,Union Square Park,"[Park, City park, Tourist attraction]"
3912,0x89c279b142e9dc41:0xf3bfe6b9d563bad,7436,Jones Beach State Park,"[State park, Amphitheater, Beach entertainment..."
2815,0x89c25b1087680443:0x178bf7c600df7a98,7293,Prospect Park,"[Park, Tourist attraction]"
3568,0x89c2607fc22166fb:0x133d2c68e8326ce1,6974,Flushing Meadows Corona Park,"[Park, Tourist attraction]"
945,0x89c25854a57fddc7:0xfc2dfa769d275c61,6575,AMC Empire 25,[Movie theater]
5258,0x89c2f37cb80715c9:0xd73f5ca83441153f,6413,New York Botanical Garden,"[Botanical garden, Educational institution, Ga..."
1325,0x89c258f55fbea537:0x49d5f074061cac4e,6356,Lincoln Center for the Performing Arts,"[Performing arts theater, Concert hall, Live m..."
4944,0x89c2cfab89e7c29b:0x3f0390ac972b85f1,6223,Bear Mountain State Park,"[State park, Tourist attraction]"
2958,0x89c25bae694479a3:0xb9949385da52e69e,6206,Barclays Center,"[Arena, Sports complex]"
450,0x89c2443594b7c71d:0x1503bf4bfd6f2c6f,6195,Luna Park in Coney Island,[Amusement park]


##### Notes
-I looked at the interaction side of our data using our current provisional activity definition. I initially sampled 100k reviews, but that made the interactions look extremely sparse. So I processed the full 18.66 million reviews instead. About 2.22 million reviews matched our provisional activity set, representing about 504k users and 12,041 reviewed activities. The median user has interacted with 3 different activities, and around 55% of users have interacted with at least 3. On the activity side, the median activity has 50 unique users and over 9,400 activities have at least 20 users. So we appear to have enough repeated user and item interactions to investigate an interaction-based recommender, although we still need to finalize the activity definition and determine appropriate filtering thresholds.

In [28]:
total_interactions = len(activity_reviews_full)

top_10 = activity_popularity.head(10)["unique_users"].sum()
top_100 = activity_popularity.head(100)["unique_users"].sum()
top_1000 = activity_popularity.head(1000)["unique_users"].sum()

print("Total review interactions:", f"{total_interactions:,}")
print()
print(
    "Top 10 activities' share:",
    f"{top_10 / total_interactions * 100:.2f}%"
)
print(
    "Top 100 activities' share:",
    f"{top_100 / total_interactions * 100:.2f}%"
)
print(
    "Top 1,000 activities' share:",
    f"{top_1000 / total_interactions * 100:.2f}%"
)

Total review interactions: 2,221,591

Top 10 activities' share: 3.02%
Top 100 activities' share: 18.77%
Top 1,000 activities' share: 58.76%


In [29]:
user_activity_interactions = (
    activity_reviews_full[
        ["user_id", "gmap_id", "rating"]
    ]
    .drop_duplicates(subset=["user_id", "gmap_id"])
    .copy()
)

print(
    "Raw activity review rows:",
    f"{len(activity_reviews_full):,}"
)

print(
    "Unique user-activity interactions:",
    f"{len(user_activity_interactions):,}"
)

print(
    "Repeated user-place reviews removed:",
    f"{len(activity_reviews_full) - len(user_activity_interactions):,}"
)

Raw activity review rows: 2,221,591
Unique user-activity interactions: 2,191,854
Repeated user-place reviews removed: 29,737


##### Notes
-The 29,737 repeated user–place reviews are only about 1.34% of the 2.22M review rows, so deduplicating leaves us with 2,191,854 unique user–activity interactions. That's still a very large interaction dataset.
Our popularity results also show a meaningful long-tail pattern: the top 10 activities account for only 3.02% of interactions, the top 100 for 18.77%, and the top 1,000 for 58.76%. So activity popularity is definitely uneven, but our dataset is not simply 10–20 extremely popular places dominating everything.

In [30]:
print("Rating distribution:")
print(
    user_activity_interactions["rating"]
    .value_counts()
    .sort_index()
)

print()
print("Rating proportions:")
print(
    user_activity_interactions["rating"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

print()
print("Average rating:", round(
    user_activity_interactions["rating"].mean(), 2
))

print("Median rating:", user_activity_interactions["rating"].median())

Rating distribution:
rating
1      42300
2      41065
3     173321
4     510149
5    1425019
Name: count, dtype: int64

Rating proportions:
rating
1     1.93
2     1.87
3     7.91
4    23.27
5    65.01
Name: proportion, dtype: float64

Average rating: 4.48
Median rating: 5.0


In [31]:
user_activity_interactions["rating"].describe()

count    2.191854e+06
mean     4.475701e+00
std      8.669236e-01
min      1.000000e+00
25%      4.000000e+00
50%      5.000000e+00
75%      5.000000e+00
max      5.000000e+00
Name: rating, dtype: float64

##### Notes
-ratings are available, but they're heavily skewed positive. About 88.3% of interactions are 4 or 5 stars. A user reviewing something doesn't necessarily mean they loved it, but in this dataset, reviews overwhelmingly represent positive experiences.
That matters because a recommender trained directly to predict star ratings could learn something, but there isn't much negative feedback compared with positive feedback. Our group should keep this in mind when deciding whether the eventual model treats ratings as:
Explicit feedback
5 stars vs. 1 star has meaning

OR

Implicit/weighted feedback
User interacted with this activity,
with rating used as an additional strength signal

##### Notes

-18.66M reviews examined.

-Using the provisional activity definition, 2.22M reviews correspond to candidate activities.

-Those reviews represent 504,136 users and 12,041 reviewed candidate activities.

-There are 2.19M unique user–activity interactions after removing repeated user/place pairs.

-Median user history = 3 unique activities.

-54.5% of users have interacted with at least 3 activities.

-30.0% have interacted with at least 5.

-Median activity = 50 unique users.

-9,421 activities have at least 20 users.

-Ratings are strongly positive-skewed: 88.28% are 4–5 stars.

-Activity popularity is also skewed, so you'll have a popularity/long-tail issue to consider when modeling.

*These numbers use the current provisional activity definition. Stacey's work will change the final activity set, and the interaction pipeline will be rerun.